# 🇲🇰 Vezilka v2 — Run Pipeline (Local + T4 GPU)

Uses your local PDFs and pipeline_v2 code, running on the connected T4 GPU.

**Estimated runtime:** ~1.5–2 hours (full 4,931 PDFs with validation)

## Step 1 — Check GPU

In [4]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("⚠️  No GPU detected — pipeline will run on CPU (much slower)")

✅ GPU: Tesla T4 (15.6 GB)
   CUDA version: 12.4
   PyTorch: 2.6.0+cu124


## Step 2 — Install Missing Packages

In [5]:
!pip install -q sacrebleu faiss-cpu laser_encoders unbabel-comet
print("✅ Missing packages installed")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 86.0 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 10.3 MB/s eta 0:00:00
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl (from fairseq>=0.12.2->laser_encoders) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/e5/f6/043b6d255dd6fbf2025110cea35b87f4c5100a181681d8eab496269f0d5b/omegaconf-2.0.5

## Step 3 — Setup Paths & Verify

In [6]:
import sys
from pathlib import Path

# Point to your pipeline_v2 code
APPROACH2_DIR = Path("/Users/edirizvani/VezillkaPipelineExctractor/pipeline_v2")
PDF_DIR = Path("/Users/edirizvani/VezillkaPipelineExctractor/pipeline_v1/data/pdfs")

# Add to Python path so imports work
if str(APPROACH2_DIR) not in sys.path:
    sys.path.insert(0, str(APPROACH2_DIR))

# Verify
assert APPROACH2_DIR.exists(), f"pipeline_v2 dir not found: {APPROACH2_DIR}"
assert PDF_DIR.exists(), f"PDF dir not found: {PDF_DIR}"

pdfs = list(PDF_DIR.rglob("*.pdf"))
print(f"✅ pipeline_v2 code: {APPROACH2_DIR}")
print(f"✅ PDFs found: {len(pdfs):,}")
print(f"   Path: {PDF_DIR}")

AssertionError: pipeline_v2 dir not found: /Users/edirizvani/VezillkaPipelineExctractor/pipeline_v2

In [7]:
# Verify all modules import correctly
from config import VezilkaConfig, DEFAULT_CONFIG
from pipeline import VezilkaPipeline
print("✅ Pipeline imports OK")

# Quick check key dependencies
import pdfplumber, fitz, sentence_transformers, laser_encoders
print(f"   pdfplumber: {pdfplumber.__version__}")
print(f"   PyMuPDF: {fitz.__doc__}")
print(f"   sentence-transformers: {sentence_transformers.__version__}")
print("   LASER3: ✅")

ModuleNotFoundError: No module named 'config'

## Step 4 — Run Pipeline 🚀

**Choose a run mode:**
- **Quick test** (10 PDFs, skip validation): ~2 min
- **Test with validation** (10 PDFs): ~10 min
- **Full run** (all PDFs): ~1.5–2 hours on T4

In [8]:
# === CHOOSE YOUR RUN MODE ===
LIMIT = 10          # Set to None for full run
SKIP_VALIDATION = True  # Set to False for full validation
# ============================

config = VezilkaConfig(
    pdf_dir=PDF_DIR,
    data_dir=APPROACH2_DIR / "data",
    extracted_dir=APPROACH2_DIR / "data" / "extracted",
    segmented_dir=APPROACH2_DIR / "data" / "segmented",
    aligned_dir=APPROACH2_DIR / "data" / "aligned",
    output_dir=APPROACH2_DIR / "data" / "output",
    failed_log=APPROACH2_DIR / "data" / "failed_pdfs.jsonl",
    skipped_log=APPROACH2_DIR / "data" / "skipped_pdfs.jsonl",
    rejected_pairs_log=APPROACH2_DIR / "data" / "rejected_pairs.jsonl",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Device: {config.device}")
print(f"PDFs: {config.pdf_dir}")
print(f"Output: {config.output_dir}")
print(f"Limit: {LIMIT or 'ALL'}")
print(f"Validation: {'SKIP' if SKIP_VALIDATION else 'FULL'}")
print()

pipeline = VezilkaPipeline(config=config)
pipeline.run(limit=LIMIT, skip_validation=SKIP_VALIDATION)

NameError: name 'VezilkaConfig' is not defined

## Step 5 — Inspect Results

In [ ]:
import pandas as pd

output_dir = APPROACH2_DIR / "data" / "output"
tsv = output_dir / "vezilka_v2_corpus.tsv"

if tsv.exists():
    df = pd.read_csv(tsv, sep="\t")
    print(f"✅ Total pairs: {len(df):,}")
    print(f"\nAlignment strategy distribution:")
    print(df['alignment_strategy'].value_counts())
    print(f"\nLayout type distribution:")
    print(df['layout_type'].value_counts())
    print(f"\nBlended confidence stats:")
    print(df['blended_confidence'].describe())
    print(f"\nTier reached:")
    print(df['tier_reached'].value_counts().sort_index())
    print(f"\n--- Sample pairs ---")
    for _, row in df.sample(min(5, len(df))).iterrows():
        print(f"\nMK: {row['mk'][:120]}")
        print(f"SQ: {row['sq'][:120]}")
        print(f"   [{row['alignment_strategy']}] blend={row['blended_confidence']:.3f}")
else:
    print("No output TSV found yet. Check logs above for errors.")

# Check log files
data_dir = APPROACH2_DIR / "data"
for log_name in ["skipped_pdfs.jsonl", "failed_pdfs.jsonl", "rejected_pairs.jsonl"]:
    log_path = data_dir / log_name
    if log_path.exists():
        count = sum(1 for _ in open(log_path))
        print(f"\n📄 {log_name}: {count:,} entries")

## Step 6 — Full Run (when ready)

Once the quick test looks good, change the settings above and re-run, or use this cell:

In [ ]:
# # === FULL RUN — uncomment and run when ready ===
# config = VezilkaConfig(
#     pdf_dir=PDF_DIR,
#     data_dir=APPROACH2_DIR / "data",
#     extracted_dir=APPROACH2_DIR / "data" / "extracted",
#     segmented_dir=APPROACH2_DIR / "data" / "segmented",
#     aligned_dir=APPROACH2_DIR / "data" / "aligned",
#     output_dir=APPROACH2_DIR / "data" / "output",
#     failed_log=APPROACH2_DIR / "data" / "failed_pdfs.jsonl",
#     skipped_log=APPROACH2_DIR / "data" / "skipped_pdfs.jsonl",
#     rejected_pairs_log=APPROACH2_DIR / "data" / "rejected_pairs.jsonl",
#     device="cuda" if torch.cuda.is_available() else "cpu",
# )
# pipeline = VezilkaPipeline(config=config)
# pipeline.run(limit=None, skip_validation=False)